In [3]:
import json, re
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qsl, urlencode, urlunparse
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os





In [4]:
hrefs = pd.read_table('G:/My Drive/GitHubProjects/MLS/data/raw/hrefs/hrefs.txt', header=None)

print(hrefs)

                                                    0
0         https://sofifa.com/team/112893/inter-miami/
1   https://sofifa.com/team/111144/seattle-sounder...
2      https://sofifa.com/team/112996/los-angeles-fc/
3              https://sofifa.com/team/697/la-galaxy/
4    https://sofifa.com/team/113018/st-louis-city-sc/
5        https://sofifa.com/team/114640/charlotte-fc/
6      https://sofifa.com/team/112885/atlanta-united/
7       https://sofifa.com/team/113149/fc-cincinnati/
8    https://sofifa.com/team/111140/portland-timbers/
9   https://sofifa.com/team/111928/san-jose-earthq...
10         https://sofifa.com/team/687/columbus-crew/
11    https://sofifa.com/team/689/new-york-red-bulls/
12  https://sofifa.com/team/691/new-england-revolu...
13        https://sofifa.com/team/698/houston-dynamo/
14    https://sofifa.com/team/112606/orlando-city-sc/
15          https://sofifa.com/team/114161/austin-fc/
16       https://sofifa.com/team/114162/nashville-sc/
17  https://sofifa.com/team/

In [6]:
def fetch_html(session, api_key, url, wait_ms=2000):
    resp = session.get(
        "https://api.scrapingfish.com/api/v1/",
        params={
            "api_key": api_key,
            "url": url,
            "js_scenario": json.dumps({
                "steps": [{"wait": wait_ms}]
            })
        },
        timeout=30
    )
    resp.raise_for_status()
    
    content_type = resp.headers.get("Content-Type", "")
    
    if "application/json" in content_type:
        payload = resp.json()
        html = payload.get("content") or payload.get("html") or payload.get("data") or ""
        if not html:
            raise ValueError(f"Empty HTML from ScrapingFish. Keys: {list(payload.keys())}")
    else:
        # API returned raw HTML directly — use it as-is
        html = resp.text
    
    if not html.strip():
        raise ValueError("Empty response")
    
    return html

all_roster_urls_2 = []

for url in hrefs.iloc[:, 0]:
    print(f"\nProcessing: {url}")
    
    session = requests.Session()

    API = 'jorXNk0XeOjcNxNdmsBHa9YXUKSwnSgMFChONEcLZh4UVsc12swTZjUE2rgfKdEgb1L7KEdEs84IhmobWb'
    
    TARGET_VERSION_TEXT = "FC 26"
    SKIP_ROSTER_DATE = 'Feb 5, 2026'
    base_url = 'https://sofifa.com'
    
    try:
        # Step 1: get the version dropdown - no click needed, options are in static HTML
        html = fetch_html(session, API, url)
        soup = BeautifulSoup(html, "lxml")

        select_season = soup.find("select", id="select-version")
        if not select_season:
            print(f"  No season dropdown. Selects found: {[s.get('id') or s.get('name') for s in soup.find_all('select')]}")
            continue

        fc26_opt = next(
            (opt for opt in select_season.find_all("option") if opt.get_text(strip=True) == TARGET_VERSION_TEXT),
            None
        )
        if not fc26_opt:
            opts = [opt.get_text(strip=True) for opt in select_season.find_all("option")]
            print(f"  No '{TARGET_VERSION_TEXT}' option. Available: {opts}")
            continue

        season_href = fc26_opt.get("value")
        if not season_href:
            print(f"  FC26 option has no value attribute")
            continue

        season_url = urljoin(base_url, season_href)
        print(f"  Season URL: {season_url}")

        # Step 2: get the roster dropdown
        roster_html = fetch_html(session, API, season_url)
        soup = BeautifulSoup(roster_html, "lxml")

        # Check both id and name since they differ in your code
        select_roster = soup.find("select", id="select-roster") or soup.find("select", attrs={"name": "roster"})
        if not select_roster:
            print(f"  No roster dropdown. Selects found: {[s.get('id') or s.get('name') for s in soup.find_all('select')]}")
            continue

        found = 0
        for option in select_roster.find_all("option"):
            roster_date = option.get_text(strip=True)
            if roster_date == SKIP_ROSTER_DATE:
                print(f"  Hit skip date '{SKIP_ROSTER_DATE}', stopping")
                break

            url_part = option.get("value", "").split("?")[0]
            if url_part:
                all_roster_urls_2.append(url_part)
                found += 1

        print(f"  Collected {found} roster URLs")

    except Exception as e:
        print(f"  FAILED: {e}")
        continue  # log and move on rather than crashing the whole loop


Processing: https://sofifa.com/team/112893/inter-miami/
  Season URL: https://sofifa.com/team/112893/260031/
  Hit skip date 'Feb 5, 2026', stopping
  Collected 10 roster URLs

Processing: https://sofifa.com/team/111144/seattle-sounders-fc/
  Season URL: https://sofifa.com/team/111144/260031/
  Hit skip date 'Feb 5, 2026', stopping
  Collected 10 roster URLs

Processing: https://sofifa.com/team/112996/los-angeles-fc/
  Season URL: https://sofifa.com/team/112996/260031/
  Hit skip date 'Feb 5, 2026', stopping
  Collected 10 roster URLs

Processing: https://sofifa.com/team/697/la-galaxy/
  Season URL: https://sofifa.com/team/697/260031/
  Hit skip date 'Feb 5, 2026', stopping
  Collected 10 roster URLs

Processing: https://sofifa.com/team/113018/st-louis-city-sc/
  Season URL: https://sofifa.com/team/113018/260031/
  Hit skip date 'Feb 5, 2026', stopping
  Collected 10 roster URLs

Processing: https://sofifa.com/team/114640/charlotte-fc/
  Season URL: https://sofifa.com/team/114640/2600

In [7]:
all_roster_urls_2026 = pd.DataFrame(all_roster_urls_2, columns=["url"])


In [8]:
all_roster_urls_2026.to_csv('G:/My Drive/GitHubProjects/MLS/data/raw/hrefs/all_roster_urls_2026.csv', index=False)

In [30]:
combined_post_26 = pd.read_csv('G:/My Drive/GitHubProjects/MLS/data/raw/hrefs/all_roster_urls_post041026.csv')

In [31]:
print('total unique roster URLs post 2026 addition:', len(combined_post_26))

total unique roster URLs post 2026 addition: 19251


In [32]:
import json

def narf_html(resp):
    txt = resp.text
    if txt.lstrip().startswith("{"):
        try:
            payload = resp.json()
            return payload.get("content") or payload.get("html") or payload.get("data") or txt
        except Exception:
            return txt
    return txt


In [34]:
import os, re, json, time
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qsl, urlencode, urlunparse
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from io import StringIO


COLS = [
"pi","ae","hi","wi","pf","oa","bo","bp","vl","wg","ta","cr","fi","he","sh","vo","ts",
"dr","cu","fr","lo","bl","to","ac","sp","ag","re","ba","tp","so","ju","st","ln","te",
"ar","in","po","vi","pe","cm","td","ma","sa","sl","tg","gd","gh","gc","gp","gr"
]

def safe_name(s: str) -> str:
    s = re.sub(r'[\\/:*?"<>|]+', '-', str(s))
    s = re.sub(r'\s+', '-', s).strip('-')
    return s

def add_columns_to_url(u: str, cols) -> str:
    pu = urlparse(u)
    pairs = parse_qsl(pu.query, keep_blank_values=True)
    pairs += [("showCol[]", c) for c in cols]
    return urlunparse(pu._replace(query=urlencode(pairs, doseq=True)))


# ---- setup ----
all_roster_urls = combined_post_26


session = requests.Session()
retry = Retry(
    total=4, backoff_factor=0.7,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
session.mount("https://", HTTPAdapter(max_retries=retry))

API = "jorXNk0XeOjcNxNdmsBHa9YXUKSwnSgMFChONEcLZh4UVsc12swTZjUE2rgfKdEgb1L7KEdEs84IhmobWb"
base_url = "https://sofifa.com"

OUTPUT_DIR = "G:/My Drive/GitHubProjects/MLS/data/raw/players/players_2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
existing_files = set(os.listdir(OUTPUT_DIR))

count = 0

for rel in all_roster_urls["url"].dropna():
    print(f"Processing roster URL: {rel}")
    base = urljoin(base_url, rel)
    scrape_url = add_columns_to_url(base, COLS)

    resp = session.get(
        "https://api.scrapingfish.com/api/v1/",
        params={"api_key": API, "url": scrape_url}
    )

    if resp.status_code != 200 or not resp.content:
        print("Bad response:", resp.status_code, "for", rel)
        continue

    soup = BeautifulSoup(resp.text, "html.parser")

    # date
    date = ""
    sel = soup.select_one("#select-roster option[selected]")
    if sel:
        date = sel.get_text(strip=True)
    date = safe_name(date)

    # team
    h1 = soup.select_one("h1")
    if not h1:
        print("No team header for:", rel)
        continue
    team = h1.get_text(strip=True)
    safe_team = safe_name(team)
    
    roster_id = rel.strip('/').split('/')[-1]
    filename = f"{safe_team}-{date}-{roster_id}.csv"

    if filename in existing_files:
        continue

    table = soup.select_one("table")
    if table is None:
        print("No table for:", rel)
        continue

    try:
        df = pd.read_html(StringIO(str(table)))[0]    
    except Exception as e:
        print("read_html failed for:", rel, "err:", e)
        continue

    df["date"] = date
    df["team"] = team

    df.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)
    existing_files.add(filename)
    count += 1
    
    print(f"Saved {count} files...")


Processing roster URL: /team/112893/260031/
Saved 1 files...
Processing roster URL: /team/112893/260030/
Saved 2 files...
Processing roster URL: /team/112893/260029/
Saved 3 files...
Processing roster URL: /team/112893/260028/
Saved 4 files...
Processing roster URL: /team/112893/260027/
Saved 5 files...
Processing roster URL: /team/112893/260026/
Saved 6 files...
Processing roster URL: /team/112893/260025/
Saved 7 files...
Processing roster URL: /team/112893/260024/
Saved 8 files...
Processing roster URL: /team/112893/260023/
Saved 9 files...
Processing roster URL: /team/112893/260022/
Saved 10 files...
Processing roster URL: /team/111144/260031/
Saved 11 files...
Processing roster URL: /team/111144/260030/
Saved 12 files...
Processing roster URL: /team/111144/260029/


KeyboardInterrupt: 

In [ ]:
"""
base_url = 'https://sofifa.com'

session = requests.Session()

API = 'jorXNk0XeOjcNxNdmsBHa9YXUKSwnSgMFChONEcLZh4UVsc12swTZjUE2rgfKdEgb1L7KEdEs84IhmobWb'

for url in hrefs.iloc[:, 0]:    
    resp = session.get("https://scraping.narf.ai/api/v1/", params={'api_key': API, 
                                                                   'url': url,
                                                                   'js_scenario': json.dumps({
                                                                       'steps': [
                                                                           {'click': "select[id='select-version']"},
                                                                           {'wait': 1000}
                                                                           ]
                                                                       })})
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    m = re.search(r'/team/(\d+)', url)
    if not m:
        print(f'could not parse {url}')
        continue
    
    team_id = m.group(1)

    select_season = soup.find('select', {'id': 'select-version'})
    if not select_season:
        print(f"No season dropdown found for url {url}")
        continue
    
    
    for season_option in select_season.find_all('option'):
        season_href = season_option.get('value')
        if not season_href:
            continue
        
        season_url = urljoin(base_url, season_href)
        
        season_name = season_option.get_text(strip=True)
        print(f"Scraping season: {season_name}")

        season_resp = session.get("https://scraping.narf.ai/api/v1/", params={"api_key": API,
                                                                              "url": season_url,
                                                                              'js_scenario': json.dumps({
                                                                                  'steps': [
                                                                                      {'click': "select[name='roster']"},
                                                                                      {'wait': 1000}
                                                                                  ]
                                                                              })})
        
        season_soup = BeautifulSoup(season_resp.text, 'html.parser')
        
        
        select_roster = season_soup.find('select', {'id': 'select-roster'})
        if not select_roster:
            print(f"No roster dropdown for season {season_name}")
            continue

        for option in select_roster.find_all('option'):
            modified_value = option.get("value")

            initial_value = modified_value.split('?')[0]

            full_url = urljoin(base_url, initial_value)
            
            print(f"Scraping initial roster URL: {full_url}")
           
            date = option.get_text(strip=True)
            safe_date = date.replace("/", "-").replace(":", "-")

            try:
                r = session.get("https://scraping.narf.ai/api/v1/", params={"api_key": API,
                                                                            "url": full_url,
                                                                            'js_scenario': json.dumps({
                                                                                'steps': [
                                                                                    {'click': "div.choices[data-type='select-multiple']"},
                                                                                    {"wait_for": ".choices-list-dropdown"},
                                                                                    {"wait_for": ".choices-list-dropdown[aria-expanded='true']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='pi']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ae']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='hi']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='wi']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='pf']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='oa']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='bo']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='bp']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='vl']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='wg']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ta']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='cr']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='fi']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='he']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='sh']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='vo']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ts']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='dr']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='cu']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='fr']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='lo']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='bl']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='to']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ac']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='sp']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ag']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='re']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ba']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='tp']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='so']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ju']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='st']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ln']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='te']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ar']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='in']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='po']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='vi']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='pe']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='cm']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='td']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='ma']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='sa']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='sl']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='tg']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='gd']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='gh']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='gc']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='gp']"},
                                                                                    {'click': ".choices-list-dropdown [data-value='gr']"},
                                                                                    {'wait': 1000}
                                                                                ]
                                                                            })})
                roster_soup = BeautifulSoup(r.text, 'html.parser')
                
                print(roster_soup)

                table = roster_soup.find('table')
                if table is None:
                    print(f'No table found for: {full_url}')
                    continue

                rows = table.find_all('tr')
                if not rows or not rows[0].find_all('th'):
                    print(f'no headers found: {full_url}')
                    continue

                headers = [th.get_text(strip=True) for th in rows[0].find_all('th')]
                data = []
                team = roster_soup.find('h1').get_text()
                safe_team = team.replace("/", "-").replace("\\", "-").replace(":", "-").strip()

                for row in rows[1:]:
                    cols = [td.get_text(strip=True) for td in row.find_all('td')]
                    if cols and len(cols) == len(headers):
                        data.append(cols)

                if not data:
                    print(f"No data rows found for: {full_url}")
                    continue

                df = pd.DataFrame(data, columns=headers)
                df['date'] = date
                df['team'] = team

                file_path = f'../data/scraping/players/{safe_team}_{safe_date}.csv'

                df.to_csv(file_path, index=False)
            except Exception as e:
                print(f'failed to scrape {full_url}, {e}')
                continue
                """